In [1]:
### heuristictesting.ipynb is a modular and agnostic notebook for testing the M heuristic on different
### horizons.

import pandas as pd
import numpy as np

def m_heuristic_accuracy(df, margin_col='margin', price_open_col='value', price_close_col='value',
                         rows_per_M=12, M_start=1, M_end=10, M_step=0.25):
    """
    Calculate heuristic accuracy across horizons.
    
    Args:
        df : DataFrame with columns for margin, open, close.
        margin_col : str, column name for margin (B - S).
        price_open_col : str, column for horizon open price.
        price_close_col : str, column for horizon close price.
        rows_per_M : int, number of intake rows per M1 (default 12 for S5 -> M1).
        M_start, M_end, M_step : horizon range.

    Returns:
        DataFrame with horizon, rows, acc_pos, acc_neg, acc_avg.
    """
    results = []
    
    def horizon_accuracy(block):
        margin_block = block[margin_col].mean()
        price_change = block[price_close_col].iloc[-1] - block[price_open_col].iloc[0]
        return np.sign(margin_block), np.sign(price_change)

    for m in np.arange(M_start, M_end + M_step, M_step):
        block_size = int(rows_per_M * m)
        
        total_blocks = total_right = 0
        total_p = total_n = 0
        right_p = right_n = 0

        for i in range(0, len(df), block_size):
            block = df.iloc[i:i+block_size]
            if len(block) < block_size:
                continue
            m_sign, p_sign = horizon_accuracy(block)
            
            total_blocks += 1

            if m_sign == p_sign:
                total_right += 1

                if p_sign > 0:
                    total_p += 1
                    right_p += 1
                elif p_sign < 0:
                    total_n += 1
                    right_n += 1
            else:
                if p_sign > 0:
                    total_p += 1
                elif p_sign < 0:
                    total_n += 1

        
        acc = total_right / total_blocks
        acc_p = right_p / total_p
        acc_n = right_n / total_n
        results.append((round(m,2), block_size,total_blocks ,  acc , acc_p , acc_n))
    
    return pd.DataFrame(results, columns=['M','RpH', 'Hcount','Acc' , 'Acc Price Up' , 'Acc Price Down'])

In [11]:
## 9/29 to 10/3 , tuned for this week... ->

trainingweek = pd.read_csv("full_labeled_USD_JPY_2025-09-29_to_2025-10-03.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12    6451  0.472640      0.102887        0.892473
1     1.25   15    5160  0.487016      0.081769        0.925689
2     1.50   18    4300  0.496512      0.061164        0.942389
3     1.75   21    3686  0.495388      0.047258        0.958791
4     2.00   24    3225  0.491163      0.035503        0.965300
..     ...  ...     ...       ...           ...             ...
112  29.00  348     222  0.509009      0.000000        1.000000
113  29.25  351     220  0.563636      0.000000        1.000000
114  29.50  354     218  0.573394      0.000000        1.000000
115  29.75  357     216  0.560185      0.000000        1.000000
116  30.00  360     215  0.595349      0.000000        1.000000

[117 rows x 6 columns]


In [12]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyoverseptember29tooct03M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.528052805280528
